**RAG**

- Components
- Sentence Transformer
- Chunking Strategies
    - Character Text Splitter
    - Recursive Character Text Splitter
    - Semantic Chunking
    - Contextual Chunking
    - Preposition Chuking

**Today's Discussion**
- Vector DB
- Retrievers

In [ ]:
!pip install -q langchain langchain_openai langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.0 MB/s eta 0:00:00


In [ ]:
import dotenv
import os
dotenv.load_dotenv("/content/.env")

True

In [ ]:
import os
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base=os.getenv("OPENROUTER_BASE_URL"),
    model_name="gpt-4o-mini-2024-07-18", # Or any model on OpenRouter
    default_headers={
        "HTTP-Referer": os.getenv("APP_URL"),
        "X-Title": os.getenv("APP_NAME"),
    }
)

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base=os.getenv("OPENROUTER_BASE_URL"),
    model="text-embedding-3-small", # Or any model on OpenRouter
    default_headers={
        "HTTP-Referer": os.getenv("APP_URL"),
        "X-Title": os.getenv("APP_NAME"),
    })

In [ ]:
!pip install -q langchain_community langchain_text_splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import os
from langchain_core.prompts import PromptTemplate


PROPOSITION_PROMPT = PromptTemplate.from_template(
    """
    You are an expert data engineer building a RAG system.
    Your task is to decompose the "Current Text" into simple, atomic propositions (facts).

    ### INPUT DATA
    1. **Document Title**: {title} (Use this for global context)
    2. **Previous Context**: {previous_window} (READ-ONLY. Use this ONLY to resolve pronouns like 'he', 'it', 'they' in the current text.)
    3. **Current Text**: {current_chunk} (EXTRACT facts from this text only.)

    ### RULES
    - **Atomic Facts**: Each sentence must be a standalone fact.
    - **Coreference Resolution**: If 'Current Text' says "He decided...", and 'Previous Context' identifies him as "Elon Musk", write "Elon Musk decided...".
    - **Isolation**: DO NOT create propositions from the 'Previous Context'. Only the 'Current Text'.

    ### OUTPUT
    Return a list of sentences separated by newlines.
    """
)

proposition_chain = PROPOSITION_PROMPT | llm

document_title = "Description about Mr.Narendra Modi (PM of India)"

from langchain_community.document_loaders import TextLoader
loader = TextLoader('/content/kb_modiji.txt')
doc = loader.load()

full_text = doc[0].page_content


from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 0,
    separators=["\n\n", "\n", " "]
    # separators=["\n\n", "\n", ".", "?", "!", " ", ""]
)

chunks = text_splitter.split_text(full_text)

final_chunk = []
window_size = 2
history = []

for i, current_chunk in enumerate(chunks):
  if not current_chunk.strip():
    continue

  previous_chunk = " ".join(history[-window_size:]) if history else "No Previous Context"
  response = proposition_chain.invoke({
      "title": document_title,
      "previous_window": previous_chunk,
      "current_chunk": current_chunk
  })
  p = response.content.split("\n")
  clean_p = [i.strip() for i in p if i.strip()]
  final_chunk.extend(clean_p)
  history.append(current_chunk)



In [ ]:
len(final_chunk)

131

In [ ]:
final_chunk

['- Narendra Modi was born on September 17, 1950, in Vadnagar, Gujarat, India.',
 "- Narendra Modi's father is Damodardas Mulchand Modi.",
 "- Narendra Modi's mother is Hiraben Modi.",
 '- Narendra Modi was born into a lower-middle-class family of the Ghanchi-Teli (oil-presser) community.',
 '- Narendra Modi is the third of six children.',
 '- Narendra Modi helped his father run a tea stall at the Vadnagar railway station.',
 '- Narendra Modi was described as an average student who was more interested in theater and debates.',
 '- Narendra Modi had an early interest in politics and public speaking.',
 '- Mr. Narendra Modi completed higher secondary education in Vadnagar.',
 "- Mr. Narendra Modi obtained a Bachelor's degree in Political Science from Delhi University in 1978 through distance learning.",
 "- Mr. Narendra Modi earned a Master's degree in Political Science from Gujarat University in 1983.",
 '- Mr. Narendra Modi was influenced by nationalist literature and ideology during h

- As of now, each chunk is a plain string
- If we want to use langchain vector database and preserve them in vector database, we need to make sure that each chunk is not a plain string but a langchain document object.
- My next step is to convert these list of strings into langchain document object.

In [ ]:
from langchain_core.documents import Document
document_to_index = []
document_title_context = "History of Narendra Modi"

for prop in final_chunk:
  d = Document(
      page_content = prop,
      metadata = {
          "source": document_title_context
      })
  document_to_index.append(d)



In [ ]:
document_to_index

[Document(metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi was born on September 17, 1950, in Vadnagar, Gujarat, India.'),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content="- Narendra Modi's father is Damodardas Mulchand Modi."),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content="- Narendra Modi's mother is Hiraben Modi."),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi was born into a lower-middle-class family of the Ghanchi-Teli (oil-presser) community.'),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi is the third of six children.'),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi helped his father run a tea stall at the Vadnagar railway station.'),
 Document(metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi was described as an average student who was more i

Vector Database is similar to your regular data but has capabilities to store high dimensional vectors insted of tables. Here also you get the CRUD functionalities.

- On RAM
- On Disk

When you are creating an application and you want the fast execution to test your RAG application, it's better to use On RAM vector database which will be efficient because you might have to delete and recreate your database multiple times.

On the other hand, once the application is built, the better way it to use the full fledge, On Disk Vector database, which is more stable, cheap and provide higher capacity than On RAM database.


- On RAM: FAISS (Facebook AI Similarty Search)
- On Disk: Chroma, LanceDB

In [ ]:
!pip install -q chromadb langchain_community langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
from langchain_chroma import Chroma

vs = Chroma.from_documents(
    documents=document_to_index,
    embedding=embeddings,
    persist_directory="famous_personalities",
    collection_name = "modi"
)


In [ ]:
vs._collection.count()

131

In [ ]:
vs.similarity_search(query="who is the father of modi?", k = 1)

[Document(id='9a72b6f0-061f-4147-826d-bc00493e0ca2', metadata={'source': 'History of Narendra Modi'}, page_content="- Narendra Modi's father is Damodardas Mulchand Modi.")]

**FAISS and MMR**

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.5 MB/s eta 0:00:00


In [ ]:
from langchain_community.vectorstores import FAISS
vs = FAISS.from_documents(
    documents = document_to_index,
    embedding=embeddings
    )

In [ ]:
ret = vs.as_retriever(search_type = 'mmr', search_kwargs = {"k": 10, "lambda_mult":0.5})

In [ ]:
query = "Government Schems, Yojanas, Initiatives or programs"
response = ret.invoke(query)

In [26]:
response

[Document(id='7558f6ae-55b8-4522-9696-8f6bdb5cf3fd', metadata={'source': 'History of Narendra Modi'}, page_content='- Mudra Yojana for small businesses is a key economic initiative.'),
 Document(id='a57df9e5-6923-4d28-8491-da9d2a8a2fca', metadata={'source': 'History of Narendra Modi'}, page_content='- Various housing schemes are in place to promote housing.'),
 Document(id='19f4c244-c3db-4e69-8f5c-7468ffc5354b', metadata={'source': 'History of Narendra Modi'}, page_content='Narendra Modi has a distinctive approach to governance.'),
 Document(id='2fa692ed-f49b-455e-b7a0-4fcc4b0e085d', metadata={'source': 'History of Narendra Modi'}, page_content='- Agricultural development programs were implemented in Gujarat.'),
 Document(id='558e1a8b-4651-437e-895d-215754b3b91d', metadata={'source': 'History of Narendra Modi'}, page_content='- Pradhan Mantri Ujjwala Yojana aims to provide LPG connections.'),
 Document(id='5db4630a-d117-4eba-9bf4-f3afaf7d64a3', metadata={'source': 'History of Narendra 

**MQR**

In [27]:
!pip install -q langchain_classic

In [28]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

mq = MultiQueryRetriever.from_llm(retriever = ret, llm = llm)

In [35]:
while True:
  query = input("Enter your query: ")
  if len(query.split(" "))<5:
    print("Write the query with more than 5 words")
    query = input("Enter your query: ")
  else:
    break

Enter your query: Who is modi
Write the query with more than 5 words
Enter your query: Government Schems by modi governemt
Enter your query: Tell me the Government Schems by modi


In [36]:
query

'Tell me the Government Schems by modi'

In [31]:
q = "Government Schems"
response = mq.invoke(q)

In [33]:
len(response)

18

In [34]:
response

[Document(id='a57df9e5-6923-4d28-8491-da9d2a8a2fca', metadata={'source': 'History of Narendra Modi'}, page_content='- Various housing schemes are in place to promote housing.'),
 Document(id='13f71b87-e9a8-41ec-bb38-c70c4a5d29ac', metadata={'source': 'History of Narendra Modi'}, page_content='- Narendra Modi emphasizes direct communication with citizens.'),
 Document(id='2e26b00f-29a1-4715-8023-e9d19274900b', metadata={'source': 'History of Narendra Modi'}, page_content='- The Citizenship Amendment Act was enacted in 2019.'),
 Document(id='3df82d01-983e-4cd0-8049-37af1f37ed5f', metadata={'source': 'History of Narendra Modi'}, page_content='- The Ayushman Bharat health insurance scheme was expanded.'),
 Document(id='0ee537e2-ebbd-48f1-9c20-38c9d38c58a1', metadata={'source': 'History of Narendra Modi'}, page_content='- The PM-KISAN program provides direct income support to farmers.'),
 Document(id='0772102f-c03e-45a6-96cc-314b6d539b55', metadata={'source': 'History of Narendra Modi'}, pa

**Let's Stich and Build the final RAG**

In [45]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


template = """Answer the question based strictly on the following context.
Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_doc(docs):
  return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {"context": mq |format_doc, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "When was he born"
response = rag_chain.invoke(query)
print(response)

Narendra Modi was born on September 17, 1950.


## Re-Ranking

In [46]:
from sentence_transformers import SentenceTransformer, CrossEncoder

In [47]:
corpus = [
    "The Eiffel Tower is in Paris",
    "The capital of France is Paris",
    "I love machine learning",
    "Football is a popular sport"
]

query = "Where is the Eiffel Tower located?"

In [48]:
bi_encoder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [49]:
corpus_embeddings = bi_encoder.encode(corpus, convert_to_tensor=True)
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)

In [50]:
corpus_embeddings

tensor([[ 0.0806,  0.0987, -0.0014,  ...,  0.0153,  0.0482,  0.0566],
        [ 0.1097,  0.0149, -0.0432,  ...,  0.0738,  0.0489,  0.0013],
        [-0.0102, -0.0923,  0.0499,  ..., -0.0170,  0.0360, -0.0783],
        [ 0.0966,  0.0281, -0.0500,  ..., -0.0255,  0.1077,  0.0603]])

In [53]:
from sentence_transformers import util
# Compute cosine similarity
scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
results = sorted(zip(corpus, scores), key=lambda x: x[1], reverse=True)

In [54]:
scores

tensor([ 0.8496,  0.2708, -0.0010,  0.0260])

In [55]:
results

[('The Eiffel Tower is in Paris', tensor(0.8496)),
 ('The capital of France is Paris', tensor(0.2708)),
 ('Football is a popular sport', tensor(0.0260)),
 ('I love machine learning', tensor(-0.0010))]

In [56]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [57]:
pairs = [[query, doc] for doc in corpus]
scores = cross_encoder.predict(pairs)
results = sorted(zip(corpus, scores), key=lambda x: x[1], reverse=True)

In [58]:
results

[('The Eiffel Tower is in Paris', np.float32(9.24313)),
 ('The capital of France is Paris', np.float32(-6.781046)),
 ('I love machine learning', np.float32(-10.948625)),
 ('Football is a popular sport', np.float32(-11.002033))]

**Summary**

- Vector DB
    - On RAM: FAISS
    - On Disk: Chroma
- Vector DB can be use as a reteriver
- Reterival Strategies
    - MMR: We are able to balance relevance to the query and diversity among the retervied documents
    - MQR: This will help us to generate multiple query for the input user query in order to improve reterival
- We stiched the entire architecture to build our first RAG
- Re-Ranking as a technique
- Comparsion of BERT with CrossEncoder